****Análise Exploratória****

**Importando as bibliotecas**

In [39]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go

**Configurando padrão visual dos gráficos**

In [4]:
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

**Carregando os dados processados**

In [5]:
df_matches = pd.read_parquet('../data/processed/matches_processed.parquet')
df_players = pd.read_parquet('../data/processed/players_processed.parquet')
df_editions = pd.read_parquet('../data/processed/editions_processed.parquet')

**Criando um Dataframe voltado para a análise do desempenho do Brasil**

***1) Obtendo Partidas que o Brasil jogou***

In [9]:
filtro_brasil = (df_matches['Home Team Name'] == 'Brazil') | (df_matches['Away Team Name'] == 'Brazil')
df_brasil_matches = df_matches[filtro_brasil].copy()

***2) Trazendo os jogadores***

In [11]:
df_brasil_completo = pd.merge(df_brasil_matches, df_players, on='MatchID', how='left')

***3) Trazendo o contexto da competição***

In [15]:
df_brasil_final = pd.merge(df_brasil_completo, df_editions[['Year', 'Winner', 'Avg Attendance']], on='Year', how='left')

**Bloco 1: Raio-X Ofensivo e Defensivo**

***Criando um DF isolando as partidas***

In [32]:
df_jogos_brasil = df_brasil_final.drop_duplicates(subset=['MatchID']).copy()
df_jogos_brasil['Year'] = df_jogos_brasil['Year'].astype(int)

***Obtendo gols marcados e sofrifos***

In [33]:
df_jogos_brasil['Gols Feitos'] = df_jogos_brasil.apply(
    lambda x: x['Home Team Goals'] if x['Home Team Name'] == 'Brazil' else x['Away Team Goals'], axis=1
)

df_jogos_brasil['Gols Sofridos'] = df_jogos_brasil.apply(
    lambda x: x['Away Team Goals'] if x['Home Team Name'] == 'Brazil' else x['Home Team Goals'], axis=1
)

***Agrupando resultados por copa***

In [35]:
desempenho_por_copa = df_jogos_brasil.groupby('Year')[['Gols Feitos', 'Gols Sofridos']].sum().reset_index()
desempenho_por_copa['Saldo de Gols'] = desempenho_por_copa['Gols Feitos'] - desempenho_por_copa['Gols Sofridos']

***Criando a visualização do Bloco 1***

In [44]:
fig = go.Figure()

# Cores vibrantes do Dark Mode original
cor_feitos = '#009B3A'   # Verde
cor_sofridos = '#002776' # Azul escuro
cor_saldo_pos = '#FEDF00'# Amarelo Ouro
cor_saldo_neg = '#FF0000'# Vermelho Alerta

# 1. Barra: Gols Feitos
fig.add_trace(go.Bar(
    x=desempenho_por_copa['Year'],
    y=desempenho_por_copa['Gols Feitos'],
    name='Gols Feitos',
    marker_color=cor_feitos,
    text=desempenho_por_copa['Gols Feitos'],
    textposition='auto',
    hovertemplate='<b>%{x}</b>: %{y} Gols Feitos<extra></extra>'
))

# 2. Barra: Gols Sofridos
fig.add_trace(go.Bar(
    x=desempenho_por_copa['Year'],
    y=desempenho_por_copa['Gols Sofridos'],
    name='Gols Sofridos',
    marker_color=cor_sofridos,
    text=desempenho_por_copa['Gols Sofridos'],
    textposition='auto',
    hovertemplate='<b>%{x}</b>: %{y} Gols Sofridos<extra></extra>'
))

# 3. Linha e Marcadores dinâmicos para o Saldo de Gols
cores_saldo = [cor_saldo_pos if saldo >= 0 else cor_saldo_neg for saldo in desempenho_por_copa['Saldo de Gols']]

fig.add_trace(go.Scatter(
    x=desempenho_por_copa['Year'],
    y=desempenho_por_copa['Saldo de Gols'],
    name='Saldo de Gols',
    mode='lines+markers+text',
    line=dict(color='white', width=2, dash='dot'),
    marker=dict(color=cores_saldo, size=12, line=dict(color='#222222', width=1)),
    text=desempenho_por_copa['Saldo de Gols'],
    textposition='top center',
    textfont=dict(color=cores_saldo, size=14, family="Arial Black"),
    hovertemplate='Saldo: %{y}<extra></extra>'
))

# ==========================================
# NOVA REPRESENTAÇÃO DA GUERRA (DISCRETA E ELEGANTE)
# ==========================================

# 1. Adicionando uma linha de seta horizontal (Bracket) no espaço vazio (na altura Y=20 por exemplo)
fig.add_shape(
    type="line",
    x0=1939, y0=20, # Início logo após a copa de 38
    x1=1949, y1=20, # Fim logo antes da copa de 50
    line=dict(color="#666666", width=2, dash="solid"),
)
# Adicionando pequenas "perninhas" verticais para fazer o formato de colchete ] [
fig.add_shape(type="line", x0=1939, y0=19, x1=1939, y1=21, line=dict(color="#666666", width=2))
fig.add_shape(type="line", x0=1949, y0=19, x1=1949, y1=21, line=dict(color="#666666", width=2))

# 2. Adicionando o texto explicativo flutuante acima da linha
fig.add_annotation(
    x=1944, # Centro do período
    y=23,   # Um pouco acima da linha
    text="Intervalo:<br>2ª Guerra Mundial",
    showarrow=False,
    font=dict(color="#AAAAAA", size=11),
    align="center"
)

# ==========================================
# CONFIGURAÇÃO DO LAYOUT
# ==========================================
fig.update_layout(
    title='<b>Raio-X: Eficiência Ofensiva e Defensiva da Seleção Brasileira</b><br><sup>Volume de gols e saldo histórico por edição</sup>',
    template='plotly_dark',
    barmode='group',
    xaxis=dict(
        title='Ano da Copa', 
        tickmode='array', 
        tickvals=desempenho_por_copa['Year'], 
        tickangle=-45,
        type='linear' 
    ),
    yaxis=dict(title='Quantidade de Gols', gridcolor='#333333', zeroline=True, zerolinecolor='#666666', zerolinewidth=2),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=40, r=40, t=100, b=40),
    hovermode="x unified",
    plot_bgcolor='#111111',
    paper_bgcolor='#111111'
)

fig.show()